# 07 · Análisis del monitoreo

**Pregunta:** ¿qué detectó el monitoreo en los 10 lotes y qué se hizo con eso?

Fuentes (exportadas con `src/exportar_evidencia.py`): `docs/monitoreo.csv`, `docs/alertas.csv`, `docs/reentrenamientos.csv`, y `data/simulacion/por_lote.csv` (`src/analisis_final.py`).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SIM = RAIZ / "data" / "simulacion"
DOCS = RAIZ / "docs"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
mon = pd.read_csv(DOCS / "monitoreo.csv")
alertas = pd.read_csv(DOCS / "alertas.csv")
print(f"{len(mon)} métricas, {len(alertas)} alertas en {mon['lote_simulacion'].nunique()} lotes")
alertas.groupby(["tipo", "metrica"]).size().rename("alertas").to_frame()

## 1. Alertas por lote (con severidad, quién la recibe y acción)

In [ ]:
cols = ["lote_simulacion", "tipo", "metrica", "variable", "valor", "umbral", "severidad", "quien_la_recibe", "accion"]
alertas[[c for c in cols if c in alertas.columns]].sort_values(["lote_simulacion", "tipo"])

## 2. Deriva de datos: PSI por variable

In [ ]:
psi = (mon[(mon["tipo"] == "datos") & (mon["metrica"] == "psi")]
         .pivot_table(index="lote_simulacion", columns="variable", values="valor"))
fig, eje = plt.subplots(figsize=(10, 4))
psi.plot(ax=eje, marker="o")
eje.axhline(0.25, color="#1E1A16", linestyle="--", label="umbral 0,25")
eje.set_title("PSI contra el entrenamiento (alerta solo si además KS/chi² da p < 0,05)")
eje.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()
psi.round(2)

`con_microscopio` cambió en todos los lotes (el microscopio se compró en agosto) y `carga_planta_14d` en varios: son cambios reales del proceso, no falsas alarmas. El modelo no usa `con_microscopio` (coeficiente 0), así que esa deriva no afecta las predicciones. En el lote 10, `generacion_levadura` y `carga_planta_14d` salen 100 % del rango del entrenamiento: la deriva inducida se detecta.

## 3. Desempeño por lote contra la CV del PSet 3 (deriva conceptual)

In [ ]:
CV_MEDIA, CV_STD = 0.635, 0.123
LIMITE = CV_MEDIA - 2 * CV_STD
lote = pd.read_csv(SIM / "por_lote.csv")
fig, eje = plt.subplots(figsize=(9, 3.5))
eje.plot(lote["lote"], lote["roc_auc"], marker="o", color="#94570F", label="ROC-AUC del lote")
eje.axhspan(CV_MEDIA - CV_STD, CV_MEDIA + CV_STD, color="#E8A33D", alpha=0.2, label="CV PSet 3: media ± 1 desv.")
eje.axhline(LIMITE, color="#1E1A16", linestyle="--", label=f"límite {LIMITE:.3f} (media − 2 desv.)")
eje.set_xlabel("lote de simulación")
eje.set_ylim(-0.05, 1.05)
eje.set_title("Lotes 7 y 8 quedan bajo el límite: el monitoreo pide reentrenar en el lote 10")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()
lote[["lote", "lotes", "merma_alta", "roc_auc"]]

## 4. Negocio: costo de los errores por lote

In [ ]:
fig, eje = plt.subplots(figsize=(9, 3.5))
for col, nombre, color in [("usd_lote_sin_accion", "sin acción (hoy)", "#BFB3A3"),
                           ("usd_lote_regla_estilo", "regla por estilo", "#7A6D60"),
                           ("usd_lote_modelo_top2", "modelo top-2", "#E8A33D"),
                           ("usd_lote_modelo_umbral", "modelo umbral fijo", "#94570F")]:
    eje.plot(lote["lote"], lote[col], marker="o", label=nombre, color=color)
eje.axvspan(9.5, 10.5, color="#D9CBB6", alpha=0.5)
eje.axvline(10.3, color="#1E1A16", linestyle=":")
eje.text(10.3, eje.get_ylim()[1] * 0.92, " reentrenamiento\n (rechazado)", fontsize=8)
eje.set_xlabel("lote de simulación")
eje.set_ylabel("USD de errores por lote")
eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Reentrenamientos

In [ ]:
re = pd.read_csv(DOCS / "reentrenamientos.csv")
cols = [c for c in ["start_time", "tags.mlflow.runName", "tags.motivo_disparo", "tags.recomendacion",
                    "metrics.champion_costo_por_lote", "metrics.challenger_costo_por_lote",
                    "metrics.champion_roc_auc", "metrics.challenger_roc_auc", "params.inicio_ventana_evaluacion"]
        if c in re.columns]
re[cols].sort_values("start_time")

**Conclusión del monitoreo:** detectó la deriva inducida del lote 10 y dos cambios reales del proceso, avisó del exceso de presupuesto en 5 semanas y de una falla real de conexión (predicción faltante en el lote 1, ya corregida). La caída del ROC-AUC en los lotes 7 y 8 disparó el reentrenamiento; el challenger costó más que el champion y se rechazó con motivo registrado.